## EDA: Temperatura diária (2014–2024, E-OBS)

Caminho do dataset:
`/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/temp_2014_2024_eobs.csv`

Objetivo: caracterizar a série (cobertura temporal, estatísticas, outliers, sazonalidade) e exportar resumos — tudo dentro de `EDA`.


### 1) Setup de bibliotecas e configuração de paths


In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

DATA_FILE = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/temp_2014_2024_eobs.csv")
assert DATA_FILE.exists(), f"CSV not found: {DATA_FILE}"

pd.options.display.float_format = "{:.3f}".format
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"Using file: {DATA_FILE}")


### 2) Leitura do CSV, parsing de datas e criação de colunas úteis

#### 2.1 Explicação do código
- `parse_dates` para `Time`.
- Tipos otimizados.
- Renomeação para nomes explícitos.
- Colunas derivadas: `temp_mean_c`, `year`, `month`.


In [ ]:
parse_dates = ["Time"]
df = pd.read_csv(
    DATA_FILE,
    parse_dates=parse_dates,
    dtype={"tx": "float32", "tn": "float32", "lat": "float32", "long": "float32"},
)

df = df.sort_values("Time").reset_index(drop=True)

column_renames = {
    "Time": "date",
    "tx": "temp_max_c",
    "tn": "temp_min_c",
    "lat": "latitude",
    "long": "longitude",
}
df = df.rename(columns=column_renames)

df["temp_mean_c"] = df[["temp_max_c", "temp_min_c"]].mean(axis=1)
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

print(df.head()); df.info()


### 3) Cobertura temporal, dias em falta, duplicados e estatísticas descritivas

#### 3.1 Explicação do código
- Frequência diária esperada e intervalo completo `full_range`.
- `missing_dates` identifica lacunas.
- `describe` com percentis.
- Duplicados por `date`.


In [ ]:
from pandas.tseries.frequencies import to_offset

start_date = df["date"].min(); end_date = df["date"].max()
expected_freq = to_offset("1D")
print(f"Start: {start_date:%Y-%m-%d}  End: {end_date:%Y-%m-%d}")

full_range = pd.date_range(start=start_date, end=end_date, freq=expected_freq)
missing_dates = full_range.difference(df["date"])
print(f"Missing days: {len(missing_dates)}")

numeric_cols = ["temp_max_c", "temp_min_c", "temp_mean_c"]
display(df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.95, 0.99]).T)

dups = df[df.duplicated("date", keep=False)]
print(f"Duplicate timestamps: {dups.shape[0]}")


### 4) Deteção de outliers (IQR e Z-score)

#### 4.1 Explicação do código
- IQR e Z-score; combinação `OR` para marcar `is_outlier_*` por variável.


In [ ]:
def detect_outliers_iqr(series: pd.Series, factor: float = 1.5) -> pd.Series:
    q1 = np.nanpercentile(series, 25)
    q3 = np.nanpercentile(series, 75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return (series < lower) | (series > upper)


def detect_outliers_zscore(series: pd.Series, threshold: float = 3.0) -> pd.Series:
    mu = np.nanmean(series)
    sigma = np.nanstd(series)
    if sigma == 0 or np.isnan(sigma):
        return pd.Series(False, index=series.index)
    z = (series - mu) / sigma
    return z.abs() > threshold

for col in numeric_cols:
    flags = detect_outliers_iqr(df[col]) | detect_outliers_zscore(df[col])
    df[f"is_outlier_{col}"] = flags
    print(f"{col}: {flags.sum()} possíveis outliers")

display(df.loc[df.filter(like="is_outlier_").any(axis=1), ["date"] + numeric_cols].head(10))


### 5A) Gráficos agregados (todas as localidades)

### 5B) Gráficos por localidade específica


In [ ]:
# Série agregada
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for idx, col in enumerate(["temp_max_c", "temp_min_c", "temp_mean_c"]):
    axes[idx].plot(df["date"], df[col], label=f"ALL SITES · {col}")
    axes[idx].scatter(df.loc[df[f"is_outlier_{col}"] , "date"], df.loc[df[f"is_outlier_{col}"] , col],
                      color="black", s=10, label="outlier")
    axes[idx].legend(loc="upper right"); axes[idx].set_ylabel(col)
axes[-1].set_xlabel("date"); plt.tight_layout(); plt.show()

# Dropdown por localidade
import ipywidgets as widgets
from IPython.display import display, clear_output

site_id = (df["latitude"].round(5).astype(str) + "_" + df["longitude"].round(5).astype(str))
df["site_id"] = site_id
sites = sorted(df["site_id"].unique())

out = widgets.Output(); dd = widgets.Dropdown(options=sites, value=sites[0], description="site:")

def plot_site(sid: str):
    d = df[df["site_id"] == sid]
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    for idx, col in enumerate(["temp_max_c", "temp_min_c", "temp_mean_c"]):
        axes[idx].plot(d["date"], d[col], label=f"{sid} · {col}")
        if f"is_outlier_{col}" in d.columns:
            m = d[f"is_outlier_{col}"]
            axes[idx].scatter(d.loc[m, "date"], d.loc[m, col], s=10, color="black", label="outlier")
        axes[idx].legend(loc="upper right"); axes[idx].set_ylabel(col)
    axes[-1].set_xlabel("date"); plt.tight_layout(); plt.show()

def on_change(c):
    if c["name"] == "value":
        with out:
            clear_output(wait=True)
            plot_site(c["new"]) 

dd.observe(on_change, names="value"); display(dd)
with out:
    plot_site(dd.value)
display(out)


### 6) Sazonalidade: climatologia diária e anomalias


In [ ]:
clim = (
    df.assign(doy=df["date"].dt.dayofyear)
      .groupby("doy")["temp_mean_c"].agg(["mean", "std"]).rename(columns={"mean": "clim_mean", "std": "clim_std"})
)

df = df.assign(doy=df["date"].dt.dayofyear).merge(clim, left_on="doy", right_index=True, how="left")
df["anomaly"] = df["temp_mean_c"] - df["clim_mean"]

fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax[0].plot(df["date"], df["temp_mean_c"], label="temp_mean")
ax[0].plot(df["date"], df["clim_mean"], label="climatology", alpha=0.8)
ax[0].fill_between(df["date"], df["clim_mean"]-df["clim_std"], df["clim_mean"]+df["clim_std"], color="gray", alpha=0.2, label="±1σ")
ax[0].legend(); ax[0].set_title("Mean temperature and daily climatology")
ax[1].plot(df["date"], df["anomaly"], color="tab:orange")
ax[1].axhline(0, color="black", lw=0.8); ax[1].set_title("Temperature anomaly (mean - climatology)")
plt.tight_layout(); plt.show()


### 7) Export de tabelas resumo (dentro de EDA)

- `temp_summary_stats.csv`
- `temp_monthly_means.csv`
- Guardados em `EDA /scripts/temperaturas/resources/`.


In [ ]:
summary_stats = df[["temp_max_c", "temp_min_c", "temp_mean_c"]].describe().T
monthly_means = df.groupby(["year", "month"])[["temp_max_c", "temp_min_c", "temp_mean_c"]].mean().reset_index()

from pathlib import Path
out_dir = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /scripts/temperaturas/resources")
out_dir.mkdir(parents=True, exist_ok=True)
summary_stats.to_csv(out_dir / "temp_summary_stats.csv")
monthly_means.to_csv(out_dir / "temp_monthly_means.csv", index=False)
print(f"Saved summaries in: {out_dir}")
